Calibration of sound level (dB) of natural speech, song and melody stimuli (thus with time-varying energy).

1. We create speech-shaped noise, so a noise signal whose spectral content matches the one of the sound material in our dataset. See procedure below.
2. Measure the SL(dB) with A noise weighting and HEQ config files for left and right ear respectively on REW + EARS Serial 860-3244
3. Which integration window should I chose? In principle is shouldn't matter now becouse we have stable energy over time.
4. The measured sound level with Psychopy volume set to 1, Windows volume set to 80% and Scarlett knob set to 3/4, and headphones in the booth is?

SL(dB) = 

In [1]:
import os
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt

In [2]:
rmse = lambda x: np.sqrt(np.mean(np.abs(x)**2))

In [5]:
# Create concatenated files
conditions = ['speech', 'song']
input_dir = './input/stimuli/'
T_crop = 60

for condition in conditions:
    files = [os.path.join(input_dir, f) for f in os.listdir(input_dir) if f.endswith(condition + '.wav')]

    # Concatenate all the audio files
    audio_data = []
    for file in files:
        data, samplerate = sf.read(file)
        print(f"Loaded {file} with samplerate {samplerate} and shape {data.shape}")

        # If the audio is stereo, take the first channel
        if len(data.shape) > 1:
            data = data[:, 0]
            data = np.squeeze(data)

        audio_data.append(data)

    concatenated_audio = np.concatenate(audio_data)

    # Get T of concatenated audio
    T = len(concatenated_audio)/samplerate
    print(f"Total duration of the audio in seconds: {T}")
    print(f"Total duration of the audio in minutes: {T/60}")

    # Compute the RMS of the concatenated audio
    rms_original = rmse(concatenated_audio)
    print(f"RMS of the concatenated audio: {rms_original}")

    # Compute fft
    S = np.fft.rfft(concatenated_audio)

    # Get magnitude and phase
    mag = np.abs(S)
    phase = np.angle(S)

    # randomize phase
    idx = np.random.permutation(len(phase))
    Sr = mag * np.exp(2* np.pi * 1j * phase[idx])

    # Invert back to time domain
    r_random = np.fft.irfft(Sr)

    # Remove first and last samples 
    r_random = r_random[10:-10]

    print(f"RMS of the reconstructed audio: {rmse(r_random)}")
    print(f"Error: {rmse(r_random) - rmse(concatenated_audio)}")

    # Crop sound file to a given length
    r_random = r_random[:int(T_crop*samplerate)]

    # Save the inverted audio
    output_file = 'calibration_sound_' + condition + '.wav'
    sf.write(output_file, r_random, samplerate)

Loaded ./input/stimuli/01_speech.wav with samplerate 44100 and shape (902484, 2)
Loaded ./input/stimuli/02_speech.wav with samplerate 44100 and shape (2048590, 2)
Loaded ./input/stimuli/03_speech.wav with samplerate 44100 and shape (3588096, 2)
Loaded ./input/stimuli/04_speech.wav with samplerate 44100 and shape (2220641, 2)
Loaded ./input/stimuli/05_speech.wav with samplerate 44100 and shape (2323167, 2)
Loaded ./input/stimuli/06_speech.wav with samplerate 44100 and shape (2365724, 2)
Loaded ./input/stimuli/08_speech.wav with samplerate 44100 and shape (2317622, 2)
Loaded ./input/stimuli/09_speech.wav with samplerate 44100 and shape (1717086, 2)
Loaded ./input/stimuli/10_speech.wav with samplerate 44100 and shape (3160534, 2)
Loaded ./input/stimuli/11_speech.wav with samplerate 44100 and shape (3143300, 2)
Loaded ./input/stimuli/12_speech.wav with samplerate 44100 and shape (1541386, 2)
Loaded ./input/stimuli/13_speech.wav with samplerate 44100 and shape (3814732, 2)
Loaded ./input/st